In [1]:
!pip install huggingface_hub pyngrok openai-whisper nest_asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=56f425d5e00a0ccbc805790dac39d5686ddc3aa0045eb4ab9e8bece206e3ef9c
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [2]:
!pip install --upgrade transformers sentence-transformers

In [3]:
!ngrok config add-authtoken 2f2i5s0cdMFS65gx7UDpaLYyieJ_73i1yKGHQNjVbzouaYPex

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Libraries**

In [5]:
# AI & Deep Learning
from sentence_transformers import SentenceTransformer
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    AutoModelForCTC, Wav2Vec2Processor,
    pipeline, AutoModelForSpeechSeq2Seq, WhisperProcessor,
    MarianMTModel, MarianTokenizer, pipeline
)
from peft import PeftModel
import torch

# Network
import asyncio
import websockets
from pyngrok import ngrok

# Utilities
import numpy as np
import json
import joblib
import functools
import struct
from sklearn.preprocessing import normalize

**Configurations and Variables**

In [6]:
# --- CONFIGURATION ---
VAD_SAMPLE_RATE = 16000
VAD_WINDOW = 512
SILENCE_THRESHOLD = 0.2
SILENCE_CHUNKS = int(SILENCE_THRESHOLD / (VAD_WINDOW / VAD_SAMPLE_RATE))
MIN_SENTENCE_LENGTH = 1.0
MAX_SENTENCE_LENGTH = 3.0
PORT = 5001

# --- DEVICE CONFIG ---
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# --- PATH VARIABLES ---
kmeans_path = "/content/drive/MyDrive/Colab Notebooks/PBL6/kmeans.joblib"
nmt_path = "/content/drive/MyDrive/Colab Notebooks/PBL6/finetune_mbart"
sentence_model_path = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
wav2vec_en_path = "pdabo1607/English_Wav2Vec_Finetune"
wav2vec_vn_path = "pdabo1607/Vietnamese_Wav2Vec_Finetune_round4"
BASE_WHISPER_MODEL = "openai/whisper-tiny"
VIE_ADAPTER_WHISPER_PATH = "/content/drive/MyDrive/Colab Notebooks/PBL6/Whisper/Whisper_Vi/whisper_tiny_vi"
EN_ADAPTER_WHISPER_PATH = "/content/drive/MyDrive/Colab Notebooks/PBL6/Whisper/Whisper_En/whisper_tiny_en"
MARIAN_EN_TO_VI = "/content/drive/MyDrive/Colab Notebooks/PBL6/MarianMT_EnVi"
MARIAN_VI_TO_EN = "/content/drive/MyDrive/Colab Notebooks/PBL6/MarianMT_ViEn"

**Load models**

In [7]:
# --- GLOBAL MODEL VARIABLES ---
vad_model = None
whisper_tiny_base_pipe = None
finetuned_whisper_vie = None
finetuned_whisper_vie_processor = None
finetuned_whisper_en = None
finetuned_whisper_en_processor = None
wav2vec_en_processor = None
wav2vec_en = None
wav2vec_vn_processor = None
wav2vec_vn = None
finetuned_translator = None
marian_en_vi = None
marian_en_vi_tokenizer = None
marian_vi_en = None
marian_vi_en_tokenizer = None
sentence_model = None
kmeans = None
vn_corrector = None
en_corrector = None


def load_models():
    global vad_model, whisper_tiny_base_pipe
    global finetuned_whisper_vie, finetuned_whisper_vie_processor
    global finetuned_whisper_en, finetuned_whisper_en_processor
    global wav2vec_en_processor, wav2vec_en, wav2vec_vn_processor, wav2vec_vn
    global finetuned_translator, sentence_model, kmeans
    global marian_en_vi, marian_en_vi_tokenizer, marian_vi_en, marian_vi_en_tokenizer
    global vn_corrector, en_corrector

    print(f"[Init] Loading models on {device}...")

    # VAD Model
    try:
        vad_model, _ = torch.hub.load('snakers4/silero-vad', 'silero_vad', force_reload=False)
        print("[Init] ✅ VAD Model")
    except Exception as e:
        print(f"[Init] ❌ VAD: {e}")

    # Whisper Base Pipeline
    try:
        whisper_tiny_base_pipe = pipeline("automatic-speech-recognition", model=BASE_WHISPER_MODEL)
        print("[Init] ✅ Whisper Tiny Base")
    except Exception as e:
        print(f"[Init] ❌ Whisper Base: {e}")

    # Whisper Finetuned Vietnamese
    try:
        base_model = AutoModelForSpeechSeq2Seq.from_pretrained(BASE_WHISPER_MODEL, torch_dtype=dtype, device_map=device)
        finetuned_whisper_vie = PeftModel.from_pretrained(base_model, VIE_ADAPTER_WHISPER_PATH).merge_and_unload()
        finetuned_whisper_vie_processor = WhisperProcessor.from_pretrained(VIE_ADAPTER_WHISPER_PATH)
        print("[Init] ✅ Whisper Vie")
    except Exception as e:
        print(f"[Init] ❌ Whisper Vie: {e}")

    # Whisper Finetuned English
    try:
        base_model = AutoModelForSpeechSeq2Seq.from_pretrained(BASE_WHISPER_MODEL, torch_dtype=dtype, device_map=device)
        finetuned_whisper_en = PeftModel.from_pretrained(base_model, EN_ADAPTER_WHISPER_PATH).merge_and_unload()
        finetuned_whisper_en_processor = WhisperProcessor.from_pretrained(EN_ADAPTER_WHISPER_PATH)
        print("[Init] ✅ Whisper En")
    except Exception as e:
        print(f"[Init] ❌ Whisper En: {e}")

    # Wav2Vec2 English
    try:
        wav2vec_en_processor = Wav2Vec2Processor.from_pretrained(wav2vec_en_path)
        wav2vec_en = AutoModelForCTC.from_pretrained(wav2vec_en_path).to(device)
        vocab_dict_en = wav2vec_en_processor.tokenizer.get_vocab()
        sorted_vocab_en = [k for k, v in sorted(vocab_dict_en.items(), key=lambda x: x[1])]
        sorted_vocab_en[vocab_dict_en["[PAD]"]] = ""
        sorted_vocab_en[vocab_dict_en["|"]] = " "
        print("[Init] ✅ Wav2Vec2 En")
    except Exception as e:
        print(f"[Init] ❌ Wav2Vec En: {e}")

    # Wav2Vec2 Vietnamese
    try:
        wav2vec_vn_processor = Wav2Vec2Processor.from_pretrained(wav2vec_vn_path)
        wav2vec_vn = AutoModelForCTC.from_pretrained(wav2vec_vn_path).to(device)
        vocab_dict_vi = wav2vec_vn_processor.tokenizer.get_vocab()
        sorted_vocab_vi = [k for k, v in sorted(vocab_dict_vi.items(), key=lambda x: x[1])]
        sorted_vocab_vi[vocab_dict_vi["[PAD]"]] = ""
        sorted_vocab_vi[vocab_dict_vi["|"]] = " "
        print("[Init] ✅ Wav2Vec2 Vn")
    except Exception as e:
        print(f"[Init] ❌ Wav2Vec Vn: {e}")

    # === Corrector===
    try:
        vn_corrector = pipeline("text2text-generation", model="bmd1905/vietnamese-correction-v2", device=device)
        en_corrector = pipeline("text2text-generation", model="oliverguhr/spelling-correction-english-base", device=device)
    except Exception as e:
        print(f"[Init Error] Corrector failed: {e}")

    # mBART Translation
    try:
        trans_model = MBartForConditionalGeneration.from_pretrained(nmt_path)
        trans_tokenizer = MBart50TokenizerFast.from_pretrained(nmt_path, use_fast=False)
        finetuned_translator = pipeline("translation", model=trans_model, tokenizer=trans_tokenizer, device=0 if device == "cuda" else -1)
        print("[Init] ✅ mBART")
    except Exception as e:
        print(f"[Init] ❌ mBART: {e}")

    # MarianMT
    try:
        marian_en_vi_tokenizer = MarianTokenizer.from_pretrained(MARIAN_EN_TO_VI)
        marian_en_vi = MarianMTModel.from_pretrained(MARIAN_EN_TO_VI).to(device)
        marian_vi_en_tokenizer = MarianTokenizer.from_pretrained(MARIAN_VI_TO_EN)
        marian_vi_en = MarianMTModel.from_pretrained(MARIAN_VI_TO_EN).to(device)
        print("[Init] ✅ MarianMT")
    except Exception as e:
        print(f"[Init] ❌ MarianMT: {e}")

    # Sentence Transformer & KMeans
    try:
        sentence_model = SentenceTransformer(sentence_model_path)
        kmeans = joblib.load(kmeans_path)
        print("[Init] ✅ Sentence Model & KMeans")
    except Exception as e:
        print(f"[Init] ❌ Sentence/KMeans: {e}")

    print("\n" + "=" * 40)
    print("MODEL STATUS:")
    print(f"VAD: {'✅' if vad_model else '❌'}")
    print(f"Wav2Vec: {'✅' if wav2vec_en and wav2vec_vn else '❌'}")
    print(f"Whisper: {'✅' if finetuned_whisper_vie and finetuned_whisper_en else '❌'}")
    print(f"Translator: {'✅' if finetuned_translator else '❌'}")
    print(f"MarianMT: {'✅' if marian_en_vi and marian_vi_en else '❌'}")
    print("=" * 40)

**Helper Functions**

In [8]:
def translate_mbart(text, is_en):
    """Translate using mBART"""
    if finetuned_translator is None:
        return text
    try:
        src_lang = "en_XX" if is_en else "vi_VN"
        tgt_lang = "vi_VN" if is_en else "en_XX"
        result = finetuned_translator(text, src_lang=src_lang, tgt_lang=tgt_lang, max_length=128)
        return result[0]['translation_text']
    except Exception as e:
        print(f"[mBART Error] {e}")
        return text


def translate_marian(text, is_en):
    """Translate using MarianMT with cluster preprocessing"""
    try:
        tokenizer = marian_en_vi_tokenizer if is_en else marian_vi_en_tokenizer
        model = marian_en_vi if is_en else marian_vi_en

        if model is None or tokenizer is None:
            return text

        embedding = normalize(sentence_model.encode([text], device='cpu'))
        cluster_id = kmeans.predict(embedding)[0]
        preprocessed_text = f"__c{cluster_id}__ {text}"

        inputs = tokenizer(preprocessed_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            translated = model.generate(**inputs, max_length=512)

        return tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
    except Exception as e:
        print(f"[MarianMT Error] {e}")
        return text

In [9]:
def wav2vec_transcribe(audio_np, origin_lang):
    """Transcribe audio using Wav2Vec2"""
    processor = wav2vec_en_processor if origin_lang == 0 else wav2vec_vn_processor
    model = wav2vec_en if origin_lang == 0 else wav2vec_vn
    corrector = en_corrector if origin_lang == 0 else vn_corrector

    inputs = processor(audio_np, sampling_rate=16000, return_tensors="pt", padding=True)
    logits = model(inputs.input_values.to(device)).logits
    text = processor.decode(torch.argmax(logits, dim=-1)[0])
    text = corrector(text, max_length=512)[0]["generated_text"]

    return text.strip() or None

In [10]:
def whisper_transcribe(audio_np, origin_lang, model_name):
    """Transcribe audio using Whisper models"""
    lang = "en" if origin_lang == 0 else "vi"

    if "tiny" in model_name:
        result = whisper_tiny_base_pipe(audio_np, generate_kwargs={"language": lang, "task": "transcribe"})
        return result["text"] if result else None

    if "finetuned" in model_name:
        processor = finetuned_whisper_en_processor if origin_lang == 0 else finetuned_whisper_vie_processor
        model = finetuned_whisper_en if origin_lang == 0 else finetuned_whisper_vie

        inputs = processor(audio_np, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device).to(dtype)

        with torch.no_grad():
            predicted_ids = model.generate(input_features, language=lang, task="transcribe", max_new_tokens=225)

        result = processor.batch_decode(predicted_ids, skip_special_tokens=True)
        return result[0] if result else None

    return None

**Main Logics**

In [11]:
def run_inference(audio_np, start_offset, duration, startClock, origin_lang, target_lang, transcription_model, translation_model):
    """ASR + Translation pipeline"""
    try:
        # Transcribe
        if transcription_model.startswith("whisper"):
            text = whisper_transcribe(audio_np, origin_lang, transcription_model)
        else:
            text = wav2vec_transcribe(audio_np, origin_lang)

        if not text:
            return None

        print(f"[ASR] {start_offset:.2f}s: '{text}'")

        # Skip translation if same language
        if target_lang == origin_lang:
            return {
                "type": "transcription",
                "text": text,
                "start": start_offset,
                "end": start_offset + duration,
                "startClock": startClock
            }

        # Translate
        is_en = origin_lang == 0
        if translation_model == "marian":
            final_text = translate_marian(text, is_en)
        else:
            final_text = translate_mbart(text, is_en)

        return {
            "type": "transcription",
            "text": final_text,
            "start": start_offset,
            "end": start_offset + duration,
            "startClock": startClock
        }

    except Exception as e:
        print(f"[Inference Error] {e}")
        return None

In [12]:
class StreamSession:
    def __init__(self, websocket, vad_model):
        self.ws = websocket
        self.vad_model = vad_model

        # Buffers & VAD State
        self.sentence_buffer = []
        self.silence_counter = 0
        self.is_speaking = False

        # Sync State
        self.anchor_video_time = 0.0
        self.samples_since_anchor = 0
        self.playback_rate = 1.0
        self.sentence_start_video_time = 0.0

        # Metadata
        self.startClock = 0.0
        self.origin_lang = 0
        self.target_lang = 0
        self.video_id = None

        # Model Configuration
        self.transcription_model = "wav2vec"
        self.translation_model = "mbart"

    def update_sync(self, data):
        msg_type = data.get('type')
        if msg_type == 'time_sync':
            self.anchor_video_time = float(data['timestamp'])
            self.samples_since_anchor = 0
        elif msg_type == 'playback_rate':
            current_offset = (self.samples_since_anchor / VAD_SAMPLE_RATE) * self.playback_rate
            self.anchor_video_time += current_offset
            self.samples_since_anchor = 0
            self.playback_rate = float(data['rate'])
        elif msg_type == 'config':
            self.transcription_model = data.get('asrModel', 'wav2vec')
            self.translation_model = data.get('translationModel', 'mbart')
        elif msg_type == 'video_changed':
            self.video_id = data.get('videoId')

    def parse_audio_message(self, message):
        self.origin_lang = message[0]
        self.target_lang = message[1]
        self.startClock = struct.unpack('<Q', message[2:10])[0]

        video_id_length = struct.unpack('<H', message[10:12])[0]
        self.video_id = message[12:12+video_id_length].decode('utf-8') if video_id_length > 0 else None

        audio_data = message[12+video_id_length:]
        audio_int16 = np.frombuffer(audio_data, dtype=np.int16)
        return torch.from_numpy(audio_int16.astype(np.float32) / 32768.0)

    def process_vad(self, chunk, current_video_time):
        vad_prob = self.vad_model(chunk.unsqueeze(0), VAD_SAMPLE_RATE).item()
        buffer_duration = (len(self.sentence_buffer) * VAD_WINDOW) / VAD_SAMPLE_RATE
        should_trigger = False

        if vad_prob > 0.7:
            if not self.is_speaking:
                self.sentence_start_video_time = current_video_time
            self.is_speaking = True
            self.silence_counter = 0
            self.sentence_buffer.append(chunk)
            if buffer_duration >= MAX_SENTENCE_LENGTH:
                should_trigger = True
        else:
            if self.is_speaking:
                self.sentence_buffer.append(chunk)
                self.silence_counter += 1
                if self.silence_counter >= SILENCE_CHUNKS and buffer_duration >= MIN_SENTENCE_LENGTH:
                    should_trigger = True

        return should_trigger

    def get_audio_package(self):
        full_audio = torch.cat(self.sentence_buffer).numpy()
        speech_duration = (len(full_audio) / VAD_SAMPLE_RATE) * self.playback_rate

        package = {
            "audio": full_audio,
            "start_time": self.sentence_start_video_time,
            "duration": speech_duration,
            "clock": self.startClock,
            "src_lang": self.origin_lang,
            "tgt_lang": self.target_lang,
            "transcription_model": self.transcription_model,
            "translation_model": self.translation_model
        }

        self.sentence_buffer = []
        self.is_speaking = False
        self.silence_counter = 0
        return package

**Processing Thread**

In [13]:
async def asr_translate_task(websocket, package):
    """Runs inference in thread pool and sends result"""
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(
        None,
        functools.partial(
            run_inference,
            package['audio'],
            package['start_time'],
            package['duration'],
            package['clock'],
            package['src_lang'],
            package['tgt_lang'],
            package['transcription_model'],
            package['translation_model']
        )
    )
    if result:
        await websocket.send(json.dumps(result))

In [14]:
async def handle_client(websocket):
    """WebSocket client handler"""
    print("[WebSocket] Client connected")
    session = StreamSession(websocket, vad_model)

    try:
        async for message in websocket:
            # Handle control messages (JSON)
            if isinstance(message, str):
                try:
                    session.update_sync(json.loads(message))
                except json.JSONDecodeError:
                    pass
                continue

            # Handle audio data (binary)
            try:
                audio_tensor = session.parse_audio_message(message)
                num_chunks = len(audio_tensor) // VAD_WINDOW

                for i in range(num_chunks):
                    chunk = audio_tensor[i * VAD_WINDOW:(i + 1) * VAD_WINDOW]
                    current_time = session.anchor_video_time + \
                        ((session.samples_since_anchor + i * VAD_WINDOW) / VAD_SAMPLE_RATE) * session.playback_rate

                    if session.process_vad(chunk, current_time):
                        package = session.get_audio_package()
                        asyncio.create_task(asr_translate_task(websocket, package))

                session.samples_since_anchor += len(audio_tensor)
            except Exception as e:
                print(f"[Error] {e}")

    except websockets.exceptions.ConnectionClosed:
        print("[WebSocket] Client disconnected")

**Main Thread**

In [ ]:
async def main():
    load_models()

    public_url = ngrok.connect(PORT).public_url
    ws_url = public_url.replace('https', 'wss').replace('http', 'ws')
    print(f"[Server] ngrok tunnel: {public_url}")
    print(f"[Server] WebSocket URL: {ws_url}")

    async with websockets.serve(handle_client, "localhost", PORT):
        await asyncio.Future()

if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError as e:
        if "running event loop" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            asyncio.run(main())
        else:
            raise

[Init] Loading models on cuda...


/usr/local/lib/python3.12/dist-packages/torch/hub.py:335: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(


Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
[Init] ✅ VAD Model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

...

[ASR] 19.01s: 'The niceest cut from.'


Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ASR] 33.76s: 'From right to last at Norvas  but that means him not.'
[WebSocket] Client connected
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[Inference Error] 'NoneType' object is not callable
[WebSocket] Client connected
[WebSocket] Client connected


`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


[ASR] 1.28s: ' Thì lưng nào thì mới có thể mua được nha ở thời điểm.'
[ASR] 16.48s: ' Nhưng mà mấy buồn năm sau thì không biết là giá nhà đã lên thế.'
[ASR] 19.68s: ' Như câu nhở, nó thuộc như câu thấp nhất trên thấp như câu'
[ASR] 22.67s: ' và mắt sâu như mình nói chỉ dành cho theo số trong khi phần đầu người lâu đầu.'
[ASR] 25.61s: ' thì vẫn đứng ngoài rất mơ sở hữu nhà.'
[ASR] 27.97s: ' Cái câu trận này đi không chỉ ở Việt Nam mà nó có khắp cả thế giới luôn.'
[ASR] 31.17s: ' Chúng ta đang ngon cuối chục là tạy thế, thế là tạy của năm sau đi không?'
[ASR] 34.26s: ' là giá nhà đã liên tinh nào rồi.'
[ASR] 20.35s: ' Như cầu nhở, nó thuộc như cầu thấp nhất trên thấp như cầu của Maslow như mình.'
[ASR] 23.40s: ' Chào tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên tên